# 01 · Pipeline de la entrega 2

**Proyecto:** Cloud Provider Analytics · Big Data ITBA 2C 2026 · grupo 7

Corre el pipeline completo (Landing → Bronze → Silver → Gold → AstraDB), una segunda corrida para mostrar la idempotencia y las pruebas, y guarda la evidencia de cada etapa en `evidence/entrega2/`. La lógica está en `src/cpa/`; este notebook la ejecuta en orden y muestra los resultados.

**Cómo correrlo** (detalle en el Quickstart del README)
- *Google Colab:* abrir este notebook desde GitHub y ejecutar todo. La primera celda instala Java 17, clona el repo e instala `requirements.txt` (PySpark 3.5.3). Para cargar AstraDB: secret `ASTRA_DB_TOKEN` en Colab Secrets y el `secure-connect-*.zip` subido a `/content`.
- *Local:* Python 3.13, Java 17 (`JAVA_HOME`), `pip install -r requirements.txt jupyter` y, para AstraDB, las variables `ASTRA_DB_TOKEN` y `ASTRA_DB_BUNDLE_PATH`. Abrir el notebook desde `notebooks/`.

Sin las credenciales de AstraDB el pipeline corre igual y solo se saltea la sección 6. Landing no se modifica: todo lo que genera el pipeline queda en las otras zonas de `datalake/`.

| Sección | Qué hace |
|---|---|
| 0 y 1 | Entorno, configuración, sesión de Spark y run log |
| 2 | CSV de Landing a Bronze |
| 3 | Eventos a Bronze con Structured Streaming (watermark, dedupe, checkpoint) |
| 4 | Reglas de calidad y Silver; lote de prueba con errores inyectados |
| 5 | Mart FinOps en Gold y anomalías de costo |
| 6 | Carga en AstraDB y consultas P1 y P2 |
| 7 | Segunda corrida completa (idempotencia) |
| 8 | Pruebas (pytest) |
| 9 | Run log, rutas y tamaños |

## 0. Entorno

In [1]:
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/jeroesquivel/bigdata-20262C-g7.git"
REPO_BRANCH = "main"   # rama que se clona en Colab

if IN_COLAB:
    # Java 17, como en el notebook de clase: Spark 3.5 soporta Java 8, 11 y 17 (D2).
    subprocess.run("apt-get update -qq && apt-get install -y -qq openjdk-17-jdk-headless > /dev/null",
                   shell=True, check=True)
    os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    REPO_DIR = Path("/content/bigdata-20262C-g7")
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", "-q", "--branch", REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)
    # Colab trae otra versión de PySpark preinstalada: se instala la del proyecto antes de importarlo.
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt")], check=True)
else:
    # En local el notebook se abre desde notebooks/: la raíz del repo es la carpeta que contiene src/cpa.
    REPO_DIR = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src" / "cpa").is_dir())

sys.path.insert(0, str(REPO_DIR / "src"))

# Secretos de AstraDB: nunca en el repo. En Colab, el token sale de Colab Secrets (ASTRA_DB_TOKEN) y el
# secure connect bundle se sube a /content (o a Drive, MyDrive/cpa/). En local, de las variables de entorno
# ASTRA_DB_TOKEN y ASTRA_DB_BUNDLE_PATH. Sin ellos el pipeline corre igual y se saltea la carga a AstraDB.
if IN_COLAB:
    from google.colab import userdata
    try:
        os.environ["ASTRA_DB_TOKEN"] = userdata.get("ASTRA_DB_TOKEN")
    except Exception as e:
        print("Sin ASTRA_DB_TOKEN en Colab Secrets:", type(e).__name__)
    bundles = sorted(Path("/content").glob("secure-connect*.zip")) + sorted(Path("/content/drive/MyDrive/cpa").glob("secure-connect*.zip"))
    if bundles:
        os.environ["ASTRA_DB_BUNDLE_PATH"] = str(bundles[0])
java = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr.splitlines()[0]
print("Colab:", IN_COLAB, "| repo:", REPO_DIR.name)
print("Python", sys.version.split()[0], "|", java, "| JAVA_HOME:", os.environ.get("JAVA_HOME"))

Colab: False | repo: bigdata-20262C-g7
Python 3.13.12 | openjdk version "17.0.21.0.3" 2026-10-24 | JAVA_HOME: /opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home


## 1. Configuración, sesión de Spark y run log

La configuración sale de `config/config.yaml` si existe, o de `config/config.example.yaml`. Las rutas del lago se resuelven contra la raíz del repo. Cada etapa del pipeline deja una fila en `_meta/run_log/run_<fecha>.jsonl` con filas leídas, escritas, rechazadas y duplicadas, duración y estado.

In [2]:
import pyspark
from cpa.config import load_config
from cpa.spark_session import get_spark
from cpa.run_log import RunLog
from cpa import lake
from cpa.serving import astra_available

cfg = load_config()
# El conector de Cassandra se agrega al crear la sesión (Spark descarga el paquete al arrancar),
# así que se decide acá si va a haber carga a AstraDB.
CON_ASTRA = astra_available()
spark = get_spark(cfg, with_cassandra=CON_ASTRA)
run_log = RunLog(cfg)

print("Spark", spark.version, "| PySpark", pyspark.__version__, "| config:", cfg.source.relative_to(REPO_DIR))
print("Zona horaria:", spark.conf.get("spark.sql.session.timeZone"),
      "| shuffle.partitions:", spark.conf.get("spark.sql.shuffle.partitions"),
      "| partitionOverwriteMode:", spark.conf.get("spark.sql.sources.partitionOverwriteMode"))
for zone, path in cfg.paths.items():
    print(f"  {zone:12s} {Path(path).relative_to(REPO_DIR) if Path(path).is_relative_to(REPO_DIR) else path}")
print("Carga a AstraDB:", "sí" if CON_ASTRA else "no (faltan ASTRA_DB_TOKEN o el bundle)")

:: loading settings :: url = jar:file:<repo>/.venv/lib/python3.13/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: ~/.ivy2/cache
The jars for the packages stored in: ~/.ivy2/jars
com.datastax.spark#spark-cassandra-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-a1e4e5b9-88e9-41f9-bcae-1572952136fa;1.0
	confs: [default]
	found com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 in central
	found com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 in central
	found org.scala-lang.modules#scala-collection-compat_2.12;2.11.0 in central
	found org.apache.cassandra#java-driver-core-shaded;4.18.1 in central
	found com.datastax.oss#native-protocol;1.5.1 in central
	found com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 in central
	found com.typesafe#config;1.4.1 in central
	found org.slf4j#slf4j-api;1.7.26 in central
	found io.dropwizard.metrics#metrics-core;4.1.18 in central
	found org.hdrhistogram#HdrHistogram;2.1.12 in central
	found org.reactivestreams#reactive-streams;1.0.3 in central
	found o

	found org.apache.commons#commons-lang3;3.10 in central
	found com.thoughtworks.paranamer#paranamer;2.8 in central
	found org.scala-lang#scala-reflect;2.12.19 in central
:: resolution report :: resolve 224ms :: artifacts dl 9ms
	:: modules in use:
	com.datastax.oss#java-driver-shaded-guava;25.1-jre-graal-sub-1 from central in [default]
	com.datastax.oss#native-protocol;1.5.1 from central in [default]
	com.datastax.spark#spark-cassandra-connector-driver_2.12;3.5.1 from central in [default]
	com.datastax.spark#spark-cassandra-connector_2.12;3.5.1 from central in [default]
	com.thoughtworks.paranamer#paranamer;2.8 from central in [default]
	com.typesafe#config;1.4.1 from central in [default]
	io.dropwizard.metrics#metrics-core;4.1.18 from central in [default]
	org.apache.cassandra#java-driver-core-shaded;4.18.1 from central in [default]
	org.apache.cassandra#java-driver-mapper-runtime;4.18.1 from central in [default]
	org.apache.cassandra#java-driver-query-builder;4.18.1 from central in [

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


Spark 3.5.3 | PySpark 3.5.3 | config: config/config.example.yaml


Zona horaria: UTC | shuffle.partitions: 8 | partitionOverwriteMode: dynamic
  landing      datalake/landing
  bronze       datalake/bronze
  silver       datalake/silver
  gold         datalake/gold
  quarantine   datalake/quarantine
  checkpoints  datalake/_checkpoints
  meta         datalake/_meta
Carga a AstraDB: sí


In [3]:
# Etapa de arranque: deja registrado el entorno de la corrida.
with run_log.stage("setup") as st:
    st["spark_version"] = spark.version
    st["python_version"] = sys.version.split()[0]
    st["java"] = java
    st["colab"] = IN_COLAB

print("run log:", Path(run_log.path).relative_to(REPO_DIR) if Path(run_log.path).is_relative_to(REPO_DIR) else run_log.path)
run_log.summary()

run log: datalake/_meta/run_log/run_20261114T002224Z.jsonl


[{'stage': 'setup',
  'status': 'ok',
  'rows_read': None,
  'rows_written': None,
  'rows_rejected': None,
  'rows_duplicated': None,
  'duration_s': 0.0}]

## 2. Ingesta batch: CSV de Landing a Bronze

Las 7 fuentes CSV se leen con su esquema explícito en modo `PERMISSIVE` y se escriben en `bronze/<fuente>/ingest_date=D/`, con `ingest_ts`, `source_file` e `ingest_date`. Las filas que no se pueden interpretar van a `quarantine/<fuente>/` con `dq_errors = [parse_error]`. Como se reemplaza la partición del día, re-ejecutar no duplica (D7).

Control de conservación: filas de Landing (contadas como líneas de texto, sin Spark) = filas en Bronze + quarantine.

In [4]:
from cpa import evidence
from cpa.bronze_batch import run_bronze_batch

resultados = run_bronze_batch(spark, cfg, run_log)
tabla_bronze = [{"fuente": r["stage"].split(".", 1)[1], "landing": r["rows_read"], "bronze": r["rows_written"],
                 "quarantine": r["rows_rejected"], "conserva": r["rows_read"] == r["rows_written"] + r["rows_rejected"],
                 "archivos": r["output_files"], "bytes": r["output_bytes"],
                 "ruta": evidence.rel(lake.zone_path(cfg, "bronze", r["stage"].split(".", 1)[1], f"ingest_date={r['ingest_date']}")),
                 "segundos": r["duration_s"]} for r in resultados]
print(evidence.markdown_table(tabla_bronze))
assert all(f["conserva"] for f in tabla_bronze), "Bronze batch no conserva las filas de Landing"
ruta = evidence.save_text(cfg, "bronze_batch", run_log.run_id,
                          f"# Bronze batch, corrida {run_log.run_id}\n\n" + evidence.markdown_table(tabla_bronze), ext="md")
print("evidencia:", evidence.rel(ruta))

| fuente | landing | bronze | quarantine | conserva | archivos | bytes | ruta | segundos |
|---|---|---|---|---|---|---|---|---|
| customers_orgs | 80 | 80 | 0 | True | 1 | 6541 | datalake/bronze/customers_orgs/ingest_date=2026-11-14 | 3.12 |
| users | 800 | 800 | 0 | True | 1 | 23797 | datalake/bronze/users/ingest_date=2026-11-14 | 0.47 |
| resources | 400 | 400 | 0 | True | 1 | 11656 | datalake/bronze/resources/ingest_date=2026-11-14 | 0.36 |
| support_tickets | 1000 | 1000 | 0 | True | 1 | 18924 | datalake/bronze/support_tickets/ingest_date=2026-11-14 | 0.37 |
| marketing_touches | 1500 | 1500 | 0 | True | 1 | 23205 | datalake/bronze/marketing_touches/ingest_date=2026-11-14 | 0.35 |
| nps_surveys | 92 | 92 | 0 | True | 1 | 3507 | datalake/bronze/nps_surveys/ingest_date=2026-11-14 | 0.31 |
| billing_monthly | 240 | 240 | 0 | True | 1 | 11774 | datalake/bronze/billing_monthly/ingest_date=2026-11-14 | 0.31 |

evidencia: evidence/entrega2/bronze_batch_20261114T002224Z.md


In [5]:
# Esquema de una fuente en Bronze: tipos del diccionario de datos más las tres columnas técnicas.
bronze_orgs = spark.read.parquet(lake.zone_path(cfg, "bronze", "customers_orgs"))
bronze_orgs.printSchema()
bronze_orgs.select("org_id", "plan_tier", "is_enterprise", "signup_date", "nps_score",
                   "source_file", "ingest_ts", "ingest_date").show(3, truncate=False)

root
 |-- org_id: string (nullable = true)
 |-- org_name: string (nullable = true)
 |-- industry: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- is_enterprise: boolean (nullable = true)
 |-- signup_date: date (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- lifecycle_stage: string (nullable = true)
 |-- marketing_source: string (nullable = true)
 |-- nps_score: double (nullable = true)
 |-- source_file: string (nullable = true)
 |-- ingest_ts: timestamp (nullable = true)
 |-- ingest_date: date (nullable = true)



+------------+---------+-------------+-----------+---------+------------------+-------------------+-----------+
|org_id      |plan_tier|is_enterprise|signup_date|nps_score|source_file       |ingest_ts          |ingest_date|
+------------+---------+-------------+-----------+---------+------------------+-------------------+-----------+
|org_xaji0y6d|standard |true         |2025-05-26 |NULL     |customers_orgs.csv|2026-11-14 00:22:24|2026-11-14 |
|org_pbhsahxt|standard |false        |2025-06-16 |-3.0     |customers_orgs.csv|2026-11-14 00:22:24|2026-11-14 |
|org_hv3a3zmf|pro      |true         |2025-05-12 |4.0      |customers_orgs.csv|2026-11-14 00:22:24|2026-11-14 |
+------------+---------+-------------+-----------+---------+------------------+-------------------+-----------+
only showing top 3 rows



## 3. Ingesta streaming: eventos a Bronze

Structured Streaming lee `usage_events_stream/` con esquema explícito, `maxFilesPerTrigger=10` y `trigger(availableNow=True)`: procesa los 120 archivos en 12 micro-lotes y termina. Cada micro-lote se escribe en `bronze/usage_events/ingest_date=D/batch_id=N/` desde `foreachBatch`, y las líneas ilegibles van a `quarantine/usage_events/`.

**Deduplicación y watermark (D5).** `dedupe_key` es el `event_id` o, si falta, un hash del archivo y del registro. La deduplicación usa `dropDuplicatesWithinWatermark` con un watermark de 1 hora sobre `ingest_ts`, la hora de ingesta. No se usa la hora del evento porque cada archivo trae eventos de los 60 días: el watermark llegaría al final del período en el primer micro-lote y casi todo lo demás quedaría como tardío. Con `ingest_ts`, ningún evento se descarta por llegar tarde: los eventos atrasados según su fecha entran cuando Silver recalcula esas fechas.

In [6]:
from cpa.bronze_stream import run_bronze_stream, checkpoint_listing

stream = run_bronze_stream(spark, cfg, run_log)
resumen_stream = {k: stream[k] for k in ("rows_read", "rows_written", "rows_rejected", "rows_duplicated",
                                         "rows_dropped_by_watermark", "micro_batches_with_data", "stream_seconds")}
print(resumen_stream)
print(evidence.markdown_table(stream["progress"]))

26/11/13 21:22:30 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/11/13 21:22:31 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


{'rows_read': 43200, 'rows_written': 43200, 'rows_rejected': 0, 'rows_duplicated': 0, 'rows_dropped_by_watermark': 0, 'micro_batches_with_data': 12, 'stream_seconds': 7.5}
| batch_id | input_rows | watermark | state_rows | dropped_by_watermark | dropped_duplicates | duration_ms | written | rejected |
|---|---|---|---|---|---|---|---|---|
| 0 | 3600 | 1970-01-01T00:00:00.000Z | 3600 | 0 | 0 | 1008 | 3600 | 0 |
| 1 | 3600 | 2026-11-13T23:22:31.092Z | 7200 | 0 | 0 | 629 | 3600 | 0 |
| 2 | 3600 | 2026-11-13T23:22:32.100Z | 10800 | 0 | 0 | 595 | 3600 | 0 |
| 3 | 3600 | 2026-11-13T23:22:32.727Z | 14400 | 0 | 0 | 571 | 3600 | 0 |
| 4 | 3600 | 2026-11-13T23:22:33.325Z | 18000 | 0 | 0 | 561 | 3600 | 0 |
| 5 | 3600 | 2026-11-13T23:22:33.895Z | 21600 | 0 | 0 | 548 | 3600 | 0 |
| 6 | 3600 | 2026-11-13T23:22:34.454Z | 25200 | 0 | 0 | 522 | 3600 | 0 |
| 7 | 3600 | 2026-11-13T23:22:35.003Z | 28800 | 0 | 0 | 547 | 3600 | 0 |
| 8 | 3600 | 2026-11-13T23:22:35.528Z | 32400 | 0 | 0 | 521 | 3600 | 0 |
| 9 

In [7]:
# Eventos fuera de orden: cada micro-lote trae eventos de todo el período (por eso el watermark va sobre ingest_ts).
from pyspark.sql import functions as F
bronze_ev = spark.read.parquet(lake.zone_path(cfg, "bronze", "usage_events"))
fuera_de_orden = (bronze_ev.groupBy("batch_id")
                  .agg(F.count("*").alias("eventos"),
                       F.date_format(F.min("timestamp"), "yyyy-MM-dd").alias("evento_mas_viejo"),
                       F.date_format(F.max("timestamp"), "yyyy-MM-dd").alias("evento_mas_nuevo"),
                       F.countDistinct(F.to_date("timestamp")).alias("dias_distintos"))
                  .orderBy("batch_id"))
fuera_de_orden.show(20)

conteo_bronze_ev = bronze_ev.count()
por_version = {str(r[0]): r[1] for r in bronze_ev.groupBy("schema_version").count().orderBy("schema_version").collect()}
print("Bronze usage_events:", conteo_bronze_ev, "| event_id distintos:", bronze_ev.select("event_id").distinct().count(),
      "| por schema_version:", por_version)
# Vale en cualquier corrida (también si el stream no tenía nada pendiente): Bronze no repite dedupe_key
# y el watermark no descartó eventos.
assert conteo_bronze_ev == bronze_ev.select("dedupe_key").distinct().count()
assert stream["rows_dropped_by_watermark"] == 0

+--------+-------+----------------+----------------+--------------+
|batch_id|eventos|evento_mas_viejo|evento_mas_nuevo|dias_distintos|
+--------+-------+----------------+----------------+--------------+
|       0|   3600|      2025-07-03|      2025-08-31|            60|
|       1|   3600|      2025-07-03|      2025-08-31|            60|
|       2|   3600|      2025-07-03|      2025-08-31|            60|
|       3|   3600|      2025-07-03|      2025-08-31|            60|
|       4|   3600|      2025-07-03|      2025-08-31|            60|
|       5|   3600|      2025-07-03|      2025-08-31|            60|
|       6|   3600|      2025-07-03|      2025-08-31|            60|
|       7|   3600|      2025-07-03|      2025-08-31|            60|
|       8|   3600|      2025-07-03|      2025-08-31|            60|
|       9|   3600|      2025-07-03|      2025-08-31|            60|
|      10|   3600|      2025-07-03|      2025-08-31|            60|
|      11|   3600|      2025-07-03|      2025-08

Bronze usage_events: 43200 | event_id distintos: 43200 | por schema_version: {'1': 10800, '2': 32400}


### 3.1 Reinicio con el mismo checkpoint

Se vuelve a lanzar el stream sin borrar nada. El checkpoint registra qué archivos ya se leyeron (`sources/`), los offsets de cada micro-lote (`offsets/`), qué lotes terminaron (`commits/`) y el estado de la deduplicación (`state/`). Por eso no hay nada pendiente: se leen 0 filas y Bronze no cambia.

In [8]:
reinicio = run_bronze_stream(spark, cfg, run_log, stage_name="bronze_stream_reinicio")
conteo_tras_reinicio = spark.read.parquet(lake.zone_path(cfg, "bronze", "usage_events")).count()
print("Filas leídas en el reinicio:", reinicio["rows_read"], "| Bronze antes:", conteo_bronze_ev, "| después:", conteo_tras_reinicio)
assert reinicio["rows_read"] == 0 and conteo_tras_reinicio == conteo_bronze_ev

chk = checkpoint_listing(cfg)
for carpeta, archivos in sorted(chk.items()):
    print(f"  {carpeta:22s} {len(archivos):3d} archivos  ej: {archivos[:3]}")

Filas leídas en el reinicio: 0 | Bronze antes: 43200 | después: 43200
  .                        1 archivos  ej: ['metadata']
  commits                 13 archivos  ej: ['0', '1', '10']
  offsets                 13 archivos  ej: ['0', '1', '10']
  sources/0               12 archivos  ej: ['0', '1', '10']
  state/0/0               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/0/_metadata      1 archivos  ej: ['schema']
  state/0/1               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/2               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/3               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/4               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/5               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/6               13 archivos  ej: ['1.delta', '10.delta', '11.delta']
  state/0/7               13 archivos  ej: ['1.delta', '10.delta', '11.delta']


26/11/13 21:22:39 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


In [9]:
# Evidencia del streaming: progreso por micro-lote, eventos fuera de orden, reinicio y checkpoint.
stats_ev = lake.path_stats(lake.zone_path(cfg, "bronze", "usage_events"))
texto = (f"# Bronze streaming, corrida {run_log.run_id}\n\n"
         f"## Resumen\n\n{evidence.markdown_table([resumen_stream])}\n"
         f"Bronze usage_events: {conteo_bronze_ev} eventos, por schema_version {por_version}; "
         f"{stats_ev['files']} archivos, {stats_ev['bytes']} bytes en `{evidence.rel(stats_ev['path'])}`.\n\n"
         f"## Progreso por micro-lote (recentProgress)\n\n{evidence.markdown_table(stream['progress'])}\n"
         f"## Eventos fuera de orden por micro-lote\n\n"
         f"{evidence.markdown_table([r.asDict() for r in fuera_de_orden.collect()])}\n"
         f"## Reinicio con el mismo checkpoint\n\nFilas leídas: {reinicio['rows_read']}. "
         f"Bronze antes: {conteo_bronze_ev}, después: {conteo_tras_reinicio}.\n\n"
         f"## Contenido del checkpoint\n\n"
         + evidence.markdown_table([{"carpeta": k, "archivos": len(v), "ejemplos": ", ".join(v[:3])} for k, v in sorted(chk.items())]))
print("evidencia:", evidence.rel(evidence.save_text(cfg, "bronze_stream", run_log.run_id, texto, ext="md")))

evidencia: evidence/entrega2/bronze_stream_20261114T002224Z.md


## 4. Calidad y Silver

Silver se recalcula completo desde Bronze en cada corrida (D7):

- `dim_org` y `dim_resource`: último snapshot de Bronze, una fila por clave natural (SCD tipo 1).
- `usage_events`: join broadcast con las dimensiones, reglas de calidad (R1 a R7 y R13), deduplicación por `event_id` (R2) y `event_date`. Se escribe particionado por `event_date`, un archivo por día.

Las reglas **marcan** en vez de descartar. Las bloqueantes (R1, R3, R4 y la métrica desconocida de R6) completan `dq_errors` y el registro va a `quarantine/silver_usage_events/`. Las demás quedan como flags en Silver: `is_negative_cost` (R5, costo menor a −0,01), `unit_imputed` (R6), `value_cast_failed` (R7) e `is_before_resource_created` (R13). Los eventos v1 conservan `carbon_kg` y `genai_tokens` en nulo (D14).

In [10]:
from cpa.silver import run_silver

silver_res = run_silver(spark, cfg, run_log)
conteos_silver = {k: silver_res["conteos"][k] for k in ("rows_read", "rows_written", "rows_rejected", "rows_duplicated", "conserva")}
print("Bronze = Silver + quarantine + duplicados:", conteos_silver)
assert conteos_silver["conserva"]
print(evidence.markdown_table(silver_res["report"]))
casos = {r["regla"]: r["casos"] for r in silver_res["report"]}
print("R5 costos < -0,01:", casos["R5_is_negative_cost"], "| R6 unit imputados:", casos["R6_unit_imputed"])

Bronze = Silver + quarantine + duplicados: {'rows_read': 43200, 'rows_written': 43200, 'rows_rejected': 0, 'rows_duplicated': 0, 'conserva': True}
| regla | tipo | accion | casos |
|---|---|---|---|
| R1_missing_event_id | bloqueante | quarantine | 0 |
| R3_invalid_timestamp | bloqueante | quarantine | 0 |
| R3_future_timestamp | bloqueante | quarantine | 0 |
| R4_orphan_org_id | bloqueante | quarantine | 0 |
| R4_orphan_resource_id | bloqueante | quarantine | 0 |
| R6_unknown_metric_unit | bloqueante | quarantine | 0 |
| R2_duplicate_event_id | deduplicación | se conserva una copia | 0 |
| R5_is_negative_cost | flag | flag; va aparte en Gold | 211 |
| R6_unit_imputed | flag | se completa desde metric | 2038 |
| R7_value_cast_failed | flag | value_num nulo | 0 |
| R13_is_before_resource_created | flag | flag | 7371 |

R5 costos < -0,01: 211 | R6 unit imputados: 2038


In [11]:
# Silver: esquema, particiones y una muestra con los flags.
silver_ev = spark.read.parquet(lake.zone_path(cfg, "silver", "usage_events"))
silver_ev.printSchema()
stats_silver = lake.path_stats(lake.zone_path(cfg, "silver", "usage_events"))
print("Particiones event_date:", silver_ev.select("event_date").distinct().count(),
      "| archivos:", stats_silver["files"], "| bytes:", stats_silver["bytes"])
print("dim_org:", silver_res["dim_org"].count(), "| dim_resource:", silver_res["dim_resource"].count())
silver_ev.where("is_negative_cost OR unit_imputed").select(
    "event_id", "event_date", "org_id", "service", "metric", "value_num", "unit", "cost_usd_increment",
    "plan_tier", "is_negative_cost", "unit_imputed").show(5, truncate=False)
print("v1 con carbon_kg no nulo:", silver_ev.where("schema_version = 1 AND carbon_kg IS NOT NULL").count())

root
 |-- event_id: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)
 |-- org_id: string (nullable = true)
 |-- resource_id: string (nullable = true)
 |-- service: string (nullable = true)
 |-- region: string (nullable = true)
 |-- metric: string (nullable = true)
 |-- value: string (nullable = true)
 |-- value_num: double (nullable = true)
 |-- unit: string (nullable = true)
 |-- cost_usd_increment: double (nullable = true)
 |-- schema_version: integer (nullable = true)
 |-- carbon_kg: double (nullable = true)
 |-- genai_tokens: long (nullable = true)
 |-- industry: string (nullable = true)
 |-- plan_tier: string (nullable = true)
 |-- hq_region: string (nullable = true)
 |-- resource_created_at: date (nullable = true)
 |-- resource_state: string (nullable = true)
 |-- is_negative_cost: boolean (nullable = true)
 |-- unit_imputed: boolean (nullable = true)
 |-- value_cast_failed: boolean (nullable = true)
 |-- is_before_resource_created: boolean (nullable = true)
 

Particiones event_date: 60 | archivos: 60 | bytes: 2896429
dim_org: 80 | dim_resource: 400
+----------------+----------+------------+--------+----------------+---------+--------+------------------+---------+----------------+------------+
|event_id        |event_date|org_id      |service |metric          |value_num|unit    |cost_usd_increment|plan_tier|is_negative_cost|unit_imputed|
+----------------+----------+------------+--------+----------------+---------+--------+------------------+---------+----------------+------------+
|evt_5z59yrbo0dec|2025-08-01|org_sg65kxvf|database|requests        |112.0    |count   |-4.4238           |standard |true            |false       |
|evt_p0o9avsdj46x|2025-08-01|org_sg65kxvf|genai   |requests        |126.0    |count   |13.9547           |standard |false           |true        |
|evt_t2sayf4lizrr|2025-08-01|org_gv0e38da|storage |storage_gb_hours|8.5823   |gb_hours|0.1811            |pro      |false           |true        |
|evt_tk3iv0pp31qu|2025-08-0

v1 con carbon_kg no nulo: 0


In [12]:
# Plan físico del join con las dimensiones: BroadcastHashJoin, sin Exchange (shuffle) de los eventos.
plan_join = evidence.explain_text(silver_res["enriched_plan"])
print(plan_join[:2500])
print("BroadcastHashJoin en el plan:", "BroadcastHashJoin" in plan_join)
print("evidencia:", evidence.rel(evidence.save_text(cfg, "explain_silver_join", run_log.run_id, plan_join)))

== Physical Plan ==
AdaptiveSparkPlan (14)
+- Project (13)
   +- BroadcastHashJoin LeftOuter BuildRight (12)
      :- Project (7)
      :  +- BroadcastHashJoin LeftOuter BuildRight (6)
      :     :- Scan parquet  (1)
      :     +- BroadcastExchange (5)
      :        +- Project (4)
      :           +- Filter (3)
      :              +- Scan parquet  (2)
      +- BroadcastExchange (11)
         +- Project (10)
            +- Filter (9)
               +- Scan parquet  (8)


(1) Scan parquet 
Output [18]: [event_id#33314, timestamp#33315, org_id#33316, resource_id#33317, service#33318, region#33319, metric#33320, value#33321, unit#33322, cost_usd_increment#33323, schema_version#33324, carbon_kg#33325, genai_tokens#33326L, source_file#33327, ingest_ts#33328-T3600000ms, dedupe_key#33329, ingest_date#33330, batch_id#33331]
Batched: true
Location: InMemoryFileIndex [file:<repo>/datalake/bronze/usage_events]
ReadSchema: struct<event_id:string,timestamp:timestamp,org_id:string,resource_id:st

In [13]:
texto = (f"# Silver y calidad, corrida {run_log.run_id}\n\n## Conservación\n\n{evidence.markdown_table([conteos_silver])}\n"
         f"## Reporte por regla\n\n{evidence.markdown_table(silver_res['report'])}\n"
         f"## Silver usage_events\n\n{silver_ev.select('event_date').distinct().count()} particiones por event_date, "
         f"{stats_silver['files']} archivos, {stats_silver['bytes']} bytes en `{evidence.rel(stats_silver['path'])}`. "
         f"dim_org: {silver_res['dim_org'].count()} filas; dim_resource: {silver_res['dim_resource'].count()} filas.\n")
print("evidencia:", evidence.rel(evidence.save_text(cfg, "silver_calidad", run_log.run_id, texto, ext="md")))

evidencia: evidence/entrega2/silver_calidad_20261114T002224Z.md


### 4.1 Lote de prueba con errores inyectados (D23)

Los datos reales no tienen errores bloqueantes, así que la quarantine queda vacía. Para mostrar que las reglas funcionan, se procesa el lote de `tests/fixtures/landing/` (descripto en `tests/fixtures/README.md`) con su propia configuración, en un lago aparte (`datalake_fixture/`). Landing no se toca.

In [14]:
from cpa.config import REPO_ROOT
from cpa.bronze_batch import run_bronze_batch
from cpa.bronze_stream import run_bronze_stream

cfg_fx = load_config(REPO_ROOT / "tests" / "fixtures" / "config_fixture.yaml")
lake.reset_lake(cfg_fx)                 # el lote de prueba siempre arranca de cero
log_fx = RunLog(cfg_fx)
bronze_fx = run_bronze_batch(spark, cfg_fx, log_fx, sources=["customers_orgs", "resources"])
stream_fx = run_bronze_stream(spark, cfg_fx, log_fx)
silver_fx = run_silver(spark, cfg_fx, log_fx)

conteos_fx = [{"etapa": r["stage"], "leidas": r["rows_read"], "escritas": r["rows_written"],
               "quarantine": r["rows_rejected"], "duplicadas": r["rows_duplicated"]} for r in log_fx.rows if r["stage"] != "silver.dims"]
print(evidence.markdown_table(conteos_fx))
print(evidence.markdown_table(silver_fx["report"]))

26/11/13 21:22:43 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


| etapa | leidas | escritas | quarantine | duplicadas |
|---|---|---|---|---|
| bronze_batch.customers_orgs | 3 | 2 | 1 | 0 |
| bronze_batch.resources | 2 | 2 | 0 | 0 |
| bronze_stream | 14 | 12 | 1 | 1 |
| silver.usage_events | 12 | 6 | 6 | 0 |

| regla | tipo | accion | casos |
|---|---|---|---|
| R1_missing_event_id | bloqueante | quarantine | 1 |
| R3_invalid_timestamp | bloqueante | quarantine | 1 |
| R3_future_timestamp | bloqueante | quarantine | 1 |
| R4_orphan_org_id | bloqueante | quarantine | 1 |
| R4_orphan_resource_id | bloqueante | quarantine | 1 |
| R6_unknown_metric_unit | bloqueante | quarantine | 1 |
| R2_duplicate_event_id | deduplicación | se conserva una copia | 0 |
| R5_is_negative_cost | flag | flag; va aparte en Gold | 1 |
| R6_unit_imputed | flag | se completa desde metric | 1 |
| R7_value_cast_failed | flag | value_num nulo | 1 |
| R13_is_before_resource_created | flag | flag | 1 |



In [15]:
# Muestras de quarantine: una por regla bloqueante, más las líneas ilegibles de Bronze.
q_silver = (spark.read.parquet(lake.zone_path(cfg_fx, "quarantine", "silver_usage_events"))
            .select("event_id", "org_id", "resource_id", "metric", F.date_format("timestamp", "yyyy-MM-dd HH:mm").alias("timestamp"),
                    F.array_join("dq_errors", ", ").alias("dq_errors"), "source_file")
            .orderBy("dq_errors"))
q_bronze = (spark.read.parquet(lake.zone_path(cfg_fx, "quarantine", "usage_events"))
            .unionByName(spark.read.parquet(lake.zone_path(cfg_fx, "quarantine", "customers_orgs")), allowMissingColumns=True)
            .select("source_file", F.substring("raw_record", 1, 70).alias("raw_record"), F.array_join("dq_errors", ", ").alias("dq_errors")))
q_silver.show(truncate=False)
q_bronze.show(truncate=False)

muestras = (f"# Muestras de quarantine (lote de prueba, D23), corrida {log_fx.run_id}\n\n"
            "Generadas con `tests/fixtures/config_fixture.yaml` sobre `tests/fixtures/landing/`, no con los datos reales.\n\n"
            f"## Conteos\n\n{evidence.markdown_table(conteos_fx)}\n"
            f"## Quarantine de Silver (reglas bloqueantes)\n\n{evidence.markdown_table([r.asDict() for r in q_silver.collect()])}\n"
            f"## Quarantine de Bronze (registros ilegibles)\n\n{evidence.markdown_table([r.asDict() for r in q_bronze.collect()])}\n"
            f"## Reporte por regla\n\n{evidence.markdown_table(silver_fx['report'])}")
print("evidencia:", evidence.rel(evidence.save_text(cfg, "quarantine_muestras", run_log.run_id, muestras, ext="md")))

+-----------------+-----------+-----------+---------+----------------+----------------------+----------------------------------------+
|event_id         |org_id     |resource_id|metric   |timestamp       |dq_errors             |source_file                             |
+-----------------+-----------+-----------+---------+----------------+----------------------+----------------------------------------+
|NULL             |org_fx1    |res_fx1    |requests |2025-08-01 10:00|R1_missing_event_id   |usage_events_stream/events_fx_0001.jsonl|
|fx_future        |org_fx1    |res_fx1    |requests |2099-01-01 00:00|R3_future_timestamp   |usage_events_stream/events_fx_0001.jsonl|
|fx_no_timestamp  |org_fx1    |res_fx1    |requests |NULL            |R3_invalid_timestamp  |usage_events_stream/events_fx_0001.jsonl|
|fx_orphan_org    |org_missing|res_fx1    |requests |2025-08-01 10:00|R4_orphan_org_id      |usage_events_stream/events_fx_0001.jsonl|
|fx_orphan_res    |org_fx1    |res_missing|requests |20

+----------------------------------------+----------------------------------------------------------------------+-----------+
|source_file                             |raw_record                                                            |dq_errors  |
+----------------------------------------+----------------------------------------------------------------------+-----------+
|usage_events_stream/events_fx_0001.jsonl|{"event_id": "fx_cut", "timestamp": "2025-08-01T10:                   |parse_error|
|customers_orgs.csv                      |org_fx3,Fixture Tres,Retail,us-east,free,False,no-es-una-fecha,rep_c,l|parse_error|
+----------------------------------------+----------------------------------------------------------------------+-----------+

evidencia: evidence/entrega2/quarantine_muestras_20261114T002224Z.md


## 5. Gold: mart FinOps `org_daily_usage_by_service`

Grano: una fila por `(org_id, usage_date, service)`. Las features son `daily_cost_usd`, `requests`, `cpu_hours`, `storage_gb_hours`, `genai_tokens` y `carbon_kg`, más `negative_cost_usd`, `negative_cost_events` y `event_count`. Los costos menores a −0,01 (R5) van en `negative_cost_usd` y no se suman en `daily_cost_usd` (A2). `genai_tokens` y `carbon_kg` quedan nulos cuando ningún evento del grupo los midió (eventos v1, D14).

En el plan físico se ve el MapReduce del diseño (sección 8): `HashAggregate` parcial (combiner), `Exchange hashpartitioning(..., 8)` (shuffle) y `HashAggregate` final (reduce).

In [16]:
from cpa.gold import run_gold

gold_res = run_gold(spark, cfg, run_log)
checks_gold = gold_res["checks"]
print(checks_gold)
assert checks_gold["grain_unique"] and checks_gold["cost_reconciles"]

mart = spark.read.parquet(lake.zone_path(cfg, "gold", "org_daily_usage_by_service"))
stats_gold = lake.path_stats(lake.zone_path(cfg, "gold", "org_daily_usage_by_service"))
print("Filas:", mart.count(), "| archivos:", stats_gold["files"], "| bytes:", stats_gold["bytes"])
mart.orderBy("org_id", "usage_date", "service").show(8)

{'rows': 11050, 'distinct_keys': 11050, 'grain_unique': True, 'gold_cost_total': 147433.9778, 'silver_cost_total': 147433.9778, 'cost_diff': 0.0, 'cost_reconciles': True, 'gold_negative_events': 211, 'silver_negative_events': 211, 'gold_event_count': 43200, 'silver_events': 43200}
Filas: 11050 | archivos: 1 | bytes: 298055
+------------+----------+---------+--------------+-----------------+--------------------+--------+---------+----------------+------------+---------+-----------+
|      org_id|usage_date|  service|daily_cost_usd|negative_cost_usd|negative_cost_events|requests|cpu_hours|storage_gb_hours|genai_tokens|carbon_kg|event_count|
+------------+----------+---------+--------------+-----------------+--------------------+--------+---------+----------------+------------+---------+-----------+
|org_0lvsnujz|2025-07-03|analytics|        8.8739|              0.0|                   0|   116.0|   4.6168|         12.7808|        NULL|     NULL|          3|
|org_0lvsnujz|2025-07-03| datab

In [17]:
plan_gold = evidence.explain_text(gold_res["mart_df"])
print(plan_gold[:3000])
print("evidencia:", evidence.rel(evidence.save_text(cfg, "explain_gold", run_log.run_id, plan_gold)))

== Physical Plan ==
AdaptiveSparkPlan (6)
+- HashAggregate (5)
   +- Exchange (4)
      +- HashAggregate (3)
         +- Project (2)
            +- Scan parquet  (1)


(1) Scan parquet 
Output [9]: [org_id#50850, service#50852, metric#50854, value_num#50856, cost_usd_increment#50858, carbon_kg#50860, genai_tokens#50861L, is_negative_cost#50867, event_date#50875]
Batched: true
Location: InMemoryFileIndex [file:<repo>/datalake/silver/usage_events]
ReadSchema: struct<org_id:string,service:string,metric:string,value_num:double,cost_usd_increment:double,carbon_kg:double,genai_tokens:bigint,is_negative_cost:boolean>

(2) Project
Output [9]: [org_id#50850, service#50852, metric#50854, value_num#50856, cost_usd_increment#50858, carbon_kg#50860, genai_tokens#50861L, is_negative_cost#50867, event_date#50875 AS usage_date#50904]
Input [9]: [org_id#50850, service#50852, metric#50854, value_num#50856, cost_usd_increment#50858, carbon_kg#50860, genai_tokens#50861L, is_negative_cost#50867, event_date

### 5.1 Componente analítico: anomalías de costo con z-score robusto (D11, D24)

Para cada par organización y servicio: `anomaly_score = 0,6745 · (x − mediana) / MAD` sobre `daily_cost_usd`, e `is_cost_anomaly` cuando el valor absoluto supera 3,5 (configurable en `analytics.anomaly_mad_threshold`). Se usa la mediana y el MAD porque los picos de costo distorsionan la media y el desvío. No hay etiquetas para entrenar un modelo, así que no se usa ML (la consigna 4.4 admite z-score, MAD o percentiles). Salida: `gold/cost_anomaly_mart`.

In [18]:
from cpa.anomalies import run_anomalies

anom = run_anomalies(spark, cfg, run_log)
st_anom = anom["stage"]
resumen_anom = {k: st_anom[k] for k in ("threshold", "rows_written", "anomalies", "anomalies_pct", "pairs", "pairs_mad_zero")}
print(resumen_anom)
print(evidence.markdown_table(st_anom["sensitivity"]))
ejemplos = (anom["scored"].where("is_cost_anomaly")
            .orderBy(F.desc(F.abs("anomaly_score")))
            .select("org_id", "usage_date", "service", "daily_cost_usd", "median_cost", "mad_cost", "anomaly_score"))
ejemplos.show(10)
print("Anomalías por servicio:")
anom["scored"].where("is_cost_anomaly").groupBy("service").count().orderBy(F.desc("count")).show()

{'threshold': 3.5, 'rows_written': 11050, 'anomalies': 130, 'anomalies_pct': 1.18, 'pairs': 262, 'pairs_mad_zero': 0}
| umbral | anomalias | pct_filas | pares_org_servicio |
|---|---|---|---|
| 2.5 | 332 | 3.0 | 147 |
| 3.5 | 130 | 1.18 | 91 |
| 5.0 | 83 | 0.75 | 69 |

+------------+----------+----------+--------------+-----------+--------+-------------+
|      org_id|usage_date|   service|daily_cost_usd|median_cost|mad_cost|anomaly_score|
+------------+----------+----------+--------------+-----------+--------+-------------+
|org_okep7y6w|2025-07-03|   compute|       202.646|    11.1426|  2.7475|      47.0133|
|org_x7eedtjv|2025-07-09|   compute|       311.255|    13.8981|  4.3827|      45.7634|
|org_okep7y6w|2025-08-07|   compute|      191.8968|    11.1426|  2.7475|      44.3744|
|org_w3zp08j3|2025-07-05|networking|       27.4203|     1.4849|  0.4557|       38.388|
|org_w3zp08j3|2025-07-16|networking|       26.4157|     1.4849|  0.4557|      36.9011|
|org_x7eedtjv|2025-08-23|   comput

In [19]:
texto = (f"# Gold, corrida {run_log.run_id}\n\n## org_daily_usage_by_service\n\n"
         f"{evidence.markdown_table([checks_gold])}\n"
         f"{stats_gold['files']} archivo(s), {stats_gold['bytes']} bytes en `{evidence.rel(stats_gold['path'])}`.\n\n"
         f"Muestra del mart:\n\n{evidence.markdown_table([r.asDict() for r in mart.orderBy('org_id', 'usage_date', 'service').limit(8).collect()])}\n"
         f"## Anomalías de costo (MAD)\n\n{evidence.markdown_table([resumen_anom])}\n"
         f"Sensibilidad al umbral:\n\n{evidence.markdown_table(st_anom['sensitivity'])}\n"
         f"Diez anomalías con mayor score:\n\n{evidence.markdown_table([r.asDict() for r in ejemplos.limit(10).collect()])}")
print("evidencia:", evidence.rel(evidence.save_text(cfg, "gold_anomalias", run_log.run_id, texto, ext="md")))

evidencia: evidence/entrega2/gold_anomalias_20261114T002224Z.md


## 6. Serving en AstraDB

Tabla `cpa.org_daily_usage_by_service` con `PRIMARY KEY ((org_id), usage_date, service)` y `usage_date` descendente (D25): P1 y P2 filtran por organización, así que cada consulta lee una sola partición y el rango de fechas se resuelve sobre la clustering key.

- El esquema (`cql/01_schema.cql`) se ejecuta con `cassandra-driver`. El keyspace `cpa` se crea desde el portal, porque AstraDB no acepta `CREATE KEYSPACE` por CQL.
- El mart se carga con spark-cassandra-connector (D9). En Cassandra un `INSERT` es un upsert por clave primaria, así que re-cargar no duplica filas.
- P1 y P2 (`cql/02_queries.cql`) se consultan con sentencias preparadas. P2 suma y ordena por servicio en el cliente (A4).

| Consulta | Tabla | Partition key | Clustering | Consistencia | Por qué |
|---|---|---|---|---|---|
| P1: costos y requests diarios por servicio en un rango de fechas | `org_daily_usage_by_service` | `org_id` | `usage_date DESC, service` | `LOCAL_QUORUM` | Una partición por organización; el rango de fechas es un corte de la clustering key |
| P2: servicios con más costo en los 14 días hasta `as_of_date` | la misma | `org_id` | `usage_date DESC, service` | `LOCAL_QUORUM` | Lee como mucho 84 filas; la suma por servicio y el top-N van en el cliente |

In [20]:
ORG_EJEMPLO = "org_53lc58dr"                 # la organización con más costo de la muestra
RANGO_P1 = ("2025-08-01", "2025-08-14")

if CON_ASTRA:
    from cpa.serving import run_serving
    serving = run_serving(spark, cfg, run_log, ORG_EJEMPLO, RANGO_P1)
    st_srv = serving["stage"]
    resumen_srv = {k: st_srv[k] for k in ("rows_read", "rows_written", "astra_equals_gold", "p1_rows", "p1_rows_gold", "p2_rows_read", "duration_s")}
    print("Gold vs AstraDB:", resumen_srv)
    assert st_srv["astra_equals_gold"] and st_srv["p1_rows"] == st_srv["p1_rows_gold"]
    print(f"\nP1 · {ORG_EJEMPLO}, {RANGO_P1[0]} a {RANGO_P1[1]} ({len(serving['p1'])} filas, primeras 12):")
    print(evidence.markdown_table(serving["p1"][:12]))
    print(f"P2 · servicios con más costo, {serving['p2']['desde']} a {serving['p2']['hasta']} ({serving['p2']['filas_leidas']} filas leídas):")
    print(evidence.markdown_table(serving["p2"]["ranking"]))
else:
    print("Se saltea: no hay credenciales de AstraDB (ver la celda 0).")

Gold vs AstraDB: {'rows_read': 11050, 'rows_written': 11050, 'astra_equals_gold': True, 'p1_rows': 64, 'p1_rows_gold': 64, 'p2_rows_read': 68, 'duration_s': 55.63}

P1 · org_53lc58dr, 2025-08-01 a 2025-08-14 (64 filas, primeras 12):
| org_id | usage_date | service | daily_cost_usd | negative_cost_usd | requests | event_count |
|---|---|---|---|---|---|---|
| org_53lc58dr | 2025-08-14 | analytics | 22.3154 | 0.0 | 344.0 | 3 |
| org_53lc58dr | 2025-08-14 | compute | 2.4244 | 0.0 | 0.0 | 3 |
| org_53lc58dr | 2025-08-14 | database | 30.5159 | 0.0 | 600.0 | 6 |
| org_53lc58dr | 2025-08-14 | genai | 44.1082 | 0.0 | 339.0 | 6 |
| org_53lc58dr | 2025-08-14 | networking | 0.0 | -1.1414 | 121.0 | 3 |
| org_53lc58dr | 2025-08-14 | storage | 2.9711 | 0.0 | 131.0 | 3 |
| org_53lc58dr | 2025-08-13 | analytics | 24.1012 | 0.0 | 319.0 | 3 |
| org_53lc58dr | 2025-08-13 | compute | 15.9698 | 0.0 | 202.0 | 3 |
| org_53lc58dr | 2025-08-13 | database | 32.8499 | 0.0 | 612.0 | 6 |
| org_53lc58dr | 2025-08-1

In [21]:
if CON_ASTRA:
    texto = (f"# Serving en AstraDB, corrida {run_log.run_id}\n\n## Carga\n\n{evidence.markdown_table([resumen_srv])}\n"
             f"## P1 · costos y requests de {ORG_EJEMPLO} entre {RANGO_P1[0]} y {RANGO_P1[1]}\n\n"
             f"Consulta: `cql/02_queries.cql`, P1. {len(serving['p1'])} filas.\n\n{evidence.markdown_table(serving['p1'])}\n"
             f"## P2 · servicios con más costo de {ORG_EJEMPLO} entre {serving['p2']['desde']} y {serving['p2']['hasta']}\n\n"
             f"Consulta: `cql/02_queries.cql`, P2. {serving['p2']['filas_leidas']} filas leídas, sumadas por servicio en el cliente.\n\n"
             f"{evidence.markdown_table(serving['p2']['ranking'])}")
    print("evidencia:", evidence.rel(evidence.save_text(cfg, "serving_astradb", run_log.run_id, texto, ext="md")))

evidencia: evidence/entrega2/serving_astradb_20261114T002224Z.md


## 7. Idempotencia: segunda corrida completa

Se vuelve a correr todo el pipeline sin borrar nada y se comparan filas y archivos de cada tabla antes y después (CE3). Cada capa evita duplicar a su manera (D7): el Bronze batch reemplaza la partición del día, el stream no tiene archivos pendientes en el checkpoint, Silver y Gold se recalculan completos y la carga a AstraDB es un upsert por clave primaria.

In [22]:
from cpa.pipeline import run_pipeline, zone_snapshot, compare_snapshots

antes = zone_snapshot(spark, cfg, with_astra=CON_ASTRA)
run_log_2 = RunLog(cfg)                      # la segunda corrida queda en su propio archivo de run_log
run_pipeline(spark, cfg, run_log_2, serving_args=(ORG_EJEMPLO, RANGO_P1) if CON_ASTRA else None)
despues = zone_snapshot(spark, cfg, with_astra=CON_ASTRA)

idempotencia = compare_snapshots(antes, despues)
print(evidence.markdown_table(idempotencia))
print("Segunda corrida:")
print(evidence.markdown_table(run_log_2.summary()))
assert all(r["sin_cambios"] for r in idempotencia), "La segunda corrida cambió alguna tabla"

texto = (f"# Idempotencia, corridas {run_log.run_id} y {run_log_2.run_id}\n\n"
         "Conteos de cada tabla después de la primera corrida y después de una segunda corrida completa, sin borrar nada.\n\n"
         f"{evidence.markdown_table(idempotencia)}\n## Etapas de la segunda corrida\n\n{evidence.markdown_table(run_log_2.summary())}")
print("evidencia:", evidence.rel(evidence.save_text(cfg, "idempotencia", run_log.run_id, texto, ext="md")))

26/11/13 21:23:49 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/11/13 21:23:50 WARN CacheManager: Asked to cache already cached data.


| tabla | filas_antes | filas_despues | archivos_antes | archivos_despues | sin_cambios |
|---|---|---|---|---|---|
| bronze/customers_orgs | 80 | 80 | 1 | 1 | True |
| bronze/users | 800 | 800 | 1 | 1 | True |
| bronze/resources | 400 | 400 | 1 | 1 | True |
| bronze/support_tickets | 1000 | 1000 | 1 | 1 | True |
| bronze/marketing_touches | 1500 | 1500 | 1 | 1 | True |
| bronze/nps_surveys | 92 | 92 | 1 | 1 | True |
| bronze/billing_monthly | 240 | 240 | 1 | 1 | True |
| bronze/usage_events | 43200 | 43200 | 12 | 12 | True |
| silver/dim_org | 80 | 80 | 1 | 1 | True |
| silver/dim_resource | 400 | 400 | 1 | 1 | True |
| silver/usage_events | 43200 | 43200 | 60 | 60 | True |
| quarantine/silver_usage_events | 0 | 0 | 1 | 1 | True |
| gold/org_daily_usage_by_service | 11050 | 11050 | 1 | 1 | True |
| gold/cost_anomaly_mart | 11050 | 11050 | 1 | 1 | True |
| gold/org_daily_usage_by_service (suma daily_cost_usd) | 148351.5759 | 148351.5759 |  |  | True |
| astradb/cpa.org_daily_usage_by_s

## 8. Pruebas automáticas

`tests/` usa pytest con una sesión local de Spark: reglas de calidad (R1, R3, R5, R6, R7), deduplicación de Silver (R2), nulos de los eventos v1, unicidad del grano y costos de Gold, el score MAD sobre una serie conocida y el lote de prueba de punta a punta.

In [23]:
prueba = subprocess.run([sys.executable, "-m", "pytest", "-q", "--color=no", "-p", "no:cacheprovider", "tests"],
                        cwd=REPO_DIR, capture_output=True, text=True)
salida_pytest = prueba.stdout[-4000:]
print(salida_pytest)
print("evidencia:", evidence.rel(evidence.save_text(cfg, "pytest", run_log.run_id, salida_pytest)))
assert prueba.returncode == 0, "Falló alguna prueba"

...........                                                              [100%]
11 passed in 12.24s

evidencia: evidence/entrega2/pytest_20261114T002224Z.txt


## 9. Run log, rutas y tamaños

El run log tiene una fila por etapa de cada corrida (`_meta/run_log/run_<fecha>.jsonl`), con filas leídas, escritas, rechazadas y duplicadas, duración y estado. Al final se listan las rutas del lago con su cantidad de archivos y su tamaño.

In [24]:
run_log_df = spark.read.json(str(Path(cfg.paths["meta"]) / "run_log" / f"run_{run_log.run_id}.jsonl"))
run_log_df.select("stage", "status", "rows_read", "rows_written", "rows_rejected", "rows_duplicated", "duration_s").show(30, truncate=False)

rutas = []
for zone in ("bronze", "silver", "gold", "quarantine", "checkpoints"):
    base = Path(cfg.paths[zone])
    for tabla in sorted(p for p in base.iterdir() if p.is_dir()) if base.exists() else []:
        s = lake.path_stats(tabla)
        rutas.append({"ruta": evidence.rel(tabla), "archivos": s["files"], "bytes": s["bytes"]})
print(evidence.markdown_table(rutas))

evidence.save_text(cfg, "run_log", run_log.run_id,
                   Path(run_log.path).read_text(encoding="utf-8") + Path(run_log_2.path).read_text(encoding="utf-8"), ext="jsonl")
print("evidencia:", evidence.rel(evidence.save_text(cfg, "tamanos_rutas", run_log.run_id,
                                                     f"# Rutas y tamaños, corrida {run_log.run_id}\n\n" + evidence.markdown_table(rutas), ext="md")))

+-------------------------------+------+---------+------------+-------------+---------------+----------+
|stage                          |status|rows_read|rows_written|rows_rejected|rows_duplicated|duration_s|
+-------------------------------+------+---------+------------+-------------+---------------+----------+
|setup                          |ok    |NULL     |NULL        |NULL         |NULL           |0.0       |
|bronze_batch.customers_orgs    |ok    |80       |80          |0            |0              |3.12      |
|bronze_batch.users             |ok    |800      |800         |0            |0              |0.47      |
|bronze_batch.resources         |ok    |400      |400         |0            |0              |0.36      |
|bronze_batch.support_tickets   |ok    |1000     |1000        |0            |0              |0.37      |
|bronze_batch.marketing_touches |ok    |1500     |1500        |0            |0              |0.35      |
|bronze_batch.nps_surveys       |ok    |92       |92   

In [25]:
spark.stop()